In [1]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from datetime import datetime
import zipfile
from dotenv import load_dotenv

In [ ]:
# setup arguments
load_dotenv()

username = os.getenv("MY_USERNAME")
password = os.getenv("MY_PASSWORD")

url_download = "https://www.datablist.com/learn/csv/download-sample-csv-files"

# 1. หา Parent Directory (ถอยกลับไป 1 ชั้นจากที่รัน script)
parent_dir = os.path.dirname(os.getcwd())

# 2. ตั้งชื่อโฟลเดอร์ตามวันที่ปัจจุบัน
today = datetime.now().strftime('%Y-%m-%d')

# 3. รวม Path ทั้งหมดเข้าด้วยกัน -> ให้ไปที่ tmp แล้วตามด้วยวันที่
# ผลลัพธ์จะได้ประมาณ: C:\Users\...\Project\tmp\2026-02-19
download_path = os.path.join(parent_dir, "tmp", today)

# 4. สร้างโฟลเดอร์ตาม Path นั้น (ถ้ายังไม่มี)
os.makedirs(download_path, exist_ok=True)

target_file = "leads-100.zip"

print(f"📁 ไฟล์จะถูกดาวน์โหลดไปที่: {download_path}")

chrome_options = Options()

# 2. add Prefs
prefs = {
    "download.default_directory": download_path, # ต้องเป็น Absolute Path
    "download.prompt_for_download": False,        # ห้ามถามที่เซฟ
    "download.directory_upgrade": True,
    "safebrowsing.enabled": False,                # ปิด Safe Browsing ชั่วคราว (เพื่อไม่ให้มันกักไฟล์)
    "plugins.always_open_pdf_externally": True,  # ปิดการเปิดไฟล์ในเบราว์เซอร์
    "profile.default_content_settings.popups": 0, # ปิด Popup ทุกชนิด
}
chrome_options.add_experimental_option("prefs", prefs)

# 3. เพิ่ม Arguments เพื่อปิดระบบความปลอดภัยที่ขวางการโหลด
chrome_options.add_argument("--disable-features=InsecureDownloadWarnings")
chrome_options.add_argument("--ignore-certificate-errors")


c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard
Path ใหม่: c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp


In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

try:
    driver.get(url_download)

    wait = WebDriverWait(driver, 10)

    # 2. ค้นหาลิงก์ 'Zip version' ของไฟล์ 'leads-100.csv'
    target_xpath = "//li[contains(., 'leads-100.csv')]//a[text()='Zip version']"
    download_btn = wait.until(EC.element_to_be_clickable((By.XPATH, target_xpath)))

    # --- ขั้นตอนเพิ่มเติมเพื่อให้การคลิกแม่นยำขึ้น ---
    
    # เลื่อนหน้าจอไปที่ปุ่มนั้นก่อน (Scroll to element)
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", download_btn)
    time.sleep(1) # รอให้หน้าจอหยุดนิ่งแป๊บนึง

    # 3. คลิกด้วยคำสั่งปกติของ Selenium
    download_btn.click()
    
    print("🚀 คลิกดาวน์โหลดไฟล์ leads-100.zip เรียบร้อยแล้ว")

    # 4. ฟังก์ชันรอจนกว่าจะโหลดเสร็จ
    def wait_for_download(directory, timeout=60):
        seconds = 0
        while seconds < timeout:
            # เช็คว่ามีไฟล์ที่นามสกุลลงท้ายด้วย .crdownload หรือ .tmp ไหม
            files = os.listdir(directory)
            if any(f.endswith('.crdownload') or f.endswith('.tmp') for f in files):
                time.sleep(1)
                seconds += 1
            else:
                # ถ้าไม่มีไฟล์ชั่วคราวแล้ว และมีไฟล์เป้าหมายปรากฏขึ้นมา
                if target_file in os.listdir(directory):
                    return True
                time.sleep(1)
                seconds += 1
        return False

    if wait_for_download(download_path):
        print("✅ ดาวน์โหลดสำเร็จ! ไฟล์เปลี่ยนจาก .tmp เป็น .zip เรียบร้อย")
    else:
        print("❌ การดาวน์โหลดใช้เวลานานเกินไป หรือเกิดข้อผิดพลาด")

finally:
    # อย่าเพิ่งรีบปิดถ้ายังไม่แน่ใจ แต่ใน try-finally จะช่วยปิด driver ให้สะอาด
    driver.quit()

🚀 คลิกดาวน์โหลดไฟล์ leads-100.zip เรียบร้อยแล้ว (Standard Click)
✅ ดาวน์โหลดสำเร็จ! ไฟล์เปลี่ยนจาก .tmp เป็น .zip เรียบร้อย


In [15]:
zip_file_path = os.path.join(download_path, "leads-100.zip") 
file_password = "your_password_here"

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # การใส่รหัสผ่านใน zipfile ต้องแปลงเป็น bytes โดยใช้ .encode()
        zip_ref.extractall(path=download_path)
    print(f"✅ แตกไฟล์ลงใน {download_path} เรียบร้อยแล้ว")

except RuntimeError as e:
    if "password" in str(e):
        print("❌ รหัสผ่านไม่ถูกต้อง")
    else:
        print(f"❌ เกิดข้อผิดพลาดขณะแตกไฟล์: {e}")
except zipfile.BadZipFile:
    print("❌ ไฟล์ ZIP เสียหรือโหลดมาไม่สมบูรณ์")
os.remove(zip_file_path)




✅ แตกไฟล์ลงใน c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp เรียบร้อยแล้ว


In [18]:
import pandas as pd

In [20]:
csv_filename = os.path.join(download_path, "leads-100.csv") 

df = pd.read_csv(csv_filename)
df.head(5)

,Index,Account Id,Lead Owner,First Name,Last Name,Company,Phone 1,Phone 2,Email 1,Email 2,Website,Source,Deal Stage,Notes
0,1,0970F99ED4a2CE4,Bethany Dixon,Victor,Cochran,"Grimes, Madden and Huff",+1-532-344-1362,2754733370,hatkinson@mclaughlin.com,ygibbs@guzman.com,http://www.rosales-molina.com/,Retargeting Ads,New Lead,Speak own such.
1,2,e9AABddbCA4AFee,Andres Callahan,Maureen,Fuentes,Crosby Inc,001-672-799-2170x5610,+1-887-577-1205x5686,bshelton@wilkins.org,hunter84@jefferson.com,https://www.myers-mendez.biz/,Google Ads,Closed Won,Consumer environment still.
2,3,99CF54fE56dDc5e,Angel Ortega,Ralph,Murillo,Velasquez-Hardin,985.939.5411x2641,+1-852-904-1856x071,mmooney@savage-shaw.com,kelsey70@leach.net,https://www.chase-miller.com/,Chatbot,Contacted,Onto goal father easy development.
3,4,5B8C661A897ACE3,Dana Mcdonald,Richard,Obrien,Barrett Ltd,223.427.4047,(240)452-2332x4601,trankaren@parks.biz,normanvickie@giles-ewing.com,http://watts.com/,Trade Show,Proposal Sent,Medical education throughout lay.
4,5,2C7aEb5e8F432Ab,Sharon Cruz,Terri,Perry,Maddox Group,4953467238,802-739-3164,katiepruitt@bauer.com,collierjasmin@barton.com,http://vincent-harvey.info/,Referral,Closed Lost,Manager poor this speak center team first.


In [21]:
print(df.shape)

(100, 14)
